# ☕ Tiny AI Coffee Doctor — M0 Baseline Pilot
**CS4085 · Team:** Yasser Alassad, Moataz Alahmadi, Nadine ElHaddad, Umar Ayyash

This notebook starts with **five-class classification only** using MobileNetV3-Small (M0). It does not yet train CBAM (M1), INT8 (M2), the coffee/non-coffee gate, or rust-severity prediction. Those are later experiments.

**Important:** The course requires topic approval and a genuine evaluation of computational efficiency. This is a *pilot*, not a final test score. The dataset may contain augmented images from the same originals, so automatically grouped splits below are only a preliminary defense against leakage, **not proof of independent test images**.

### Dataset download (official pages)
- [JMuBEN — Rust, Cercospora, Phoma](https://data.mendeley.com/datasets/t2r6rszp5c/1)
- [JMuBEN2 — Healthy, Miner](https://data.mendeley.com/datasets/tgv3zb82nd/1)
- [RoCoLe — later severity stage](https://data.mendeley.com/datasets/c5yvn32dzg/2)

**Do not upload RoCoLe yet.** Download and extract the first two datasets and organize the five class folders as shown in the next cell. If the original ZIPs have nested folders, manually move the images into the correct class folder. Keep an untouched backup of the originals.

## Step 1 — Prepare folders in Google Drive
Create a folder in **My Drive** named `CoffeeDoctor/data` containing exactly these subfolders:

```
CoffeeDoctor/
  data/
    Healthy/       # JMuBEN2
    Miner/         # JMuBEN2
    Rust/          # JMuBEN
    Cercospora/    # JMuBEN
    Phoma/         # JMuBEN
```

Each subfolder should contain the actual image files (nested image subfolders are okay for the scanner below). Do **not** place zip files directly inside these class folders.

Then open this notebook in Colab, choose **Runtime → Change runtime type → T4 GPU** if offered, and run cells in order.

In [ ]:
# Colab environment / libraries
!pip -q install imagehash scikit-learn pandas pillow tqdm
import os, time, hashlib, json, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
import imagehash
import torch, torchvision
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| device:', DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT=Path('/content/drive/MyDrive/CoffeeDoctor/data')
CLASSES=['Healthy','Miner','Rust','Cercospora','Phoma']
IMAGE_EXT={'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
missing=[c for c in CLASSES if not (DATA_ROOT/c).is_dir()]
if missing:
    raise FileNotFoundError(f'Missing class folders: {missing}. Expected under {DATA_ROOT}')
rows=[]
for label in CLASSES:
    for path in (DATA_ROOT/label).rglob('*'):
        if path.is_file() and path.suffix.lower() in IMAGE_EXT:
            rows.append({'path':str(path),'label':label})
df=pd.DataFrame(rows)
if df.empty or set(df.label)!=set(CLASSES):
    raise ValueError('All five class folders must contain images before continuing.')
print(df.groupby('label').size().to_string())
print('Total files:', len(df))

## Step 2 — Inspect duplicates and create a provisional group-aware split

The scan below computes a **rotation/flip-invariant perceptual hash** for each image and groups images with identical hashes. This helps catch some transformed duplicates. It **does not** catch every crop, recoloring, or near-duplicate. The team must still review dataset metadata, naming conventions and representative images, and ideally group by known original-photo ID before claiming final test accuracy.

**First run tip:** Start with `MAX_PER_CLASS = 250` for a fast pilot. Later change to `None` to audit the full dataset (can take longer). This notebook reports **pilot** results only.

In [ ]:
MAX_PER_CLASS=250  # change to None for full scan after the pilot
sampled=pd.concat([
    df[df.label==c].sample(n=min(int((df.label==c).sum()),MAX_PER_CLASS),random_state=SEED)
    if MAX_PER_CLASS is not None else df[df.label==c]
    for c in CLASSES
],ignore_index=True)

def canonical_phash(path):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        # Identical pHash under common flips and right-angle rotations.
        hashes=[]
        for angle in (0,90,180,270):
            rotated=im.rotate(angle,expand=True)
            hashes.append(str(imagehash.phash(rotated)))
            hashes.append(str(imagehash.phash(ImageOps.mirror(rotated))))
        return min(hashes)

valid=[]; broken=[]
for r in tqdm(sampled.itertuples(index=False),total=len(sampled),desc='Hashing images'):
    try: valid.append({'path':r.path,'label':r.label,'group':canonical_phash(r.path)})
    except (UnidentifiedImageError,OSError,ValueError) as e:
        broken.append((r.path,str(e)))
audit=pd.DataFrame(valid)
print('Readable images:',len(audit),'| unreadable:',len(broken))
print('Perceptual-hash groups:',audit.group.nunique())
print('Grouped images (possible duplicates):',int(audit.group.duplicated(keep=False).sum()))
conflicts=audit.groupby('group').label.nunique()
print('Groups spanning different labels (needs manual review):',int((conflicts>1).sum()))
if (conflicts>1).any():
    raise ValueError('Potential cross-class duplicate groups found. Inspect before training.')
if audit.label.nunique()!=5 or audit.groupby('label').group.nunique().min()<5:
    raise ValueError('Not enough valid independent groups in each class for a split.')

## Step 3 — Group-aware train/validation/test partition
We use stratified group folds so identical perceptual-hash groups cannot cross partitions. This is **still provisional** until the team verifies all original/augmented relationships. Test is reserved for the **final** evaluation and is not used during this pilot.

In [ ]:
y=audit.label.to_numpy(); groups=audit.group.to_numpy()
folds=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
trainval_idx,test_idx=next(folds.split(audit,y,groups))
trainval=audit.iloc[trainval_idx].reset_index(drop=True)
inner=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED+1)
tr_idx,va_idx=next(inner.split(trainval,trainval.label,trainval.group))
train_df=trainval.iloc[tr_idx].reset_index(drop=True)
val_df=trainval.iloc[va_idx].reset_index(drop=True)
test_df=audit.iloc[test_idx].reset_index(drop=True)
assert not(set(train_df.group)&set(val_df.group))
assert not(set(train_df.group)&set(test_df.group))
assert not(set(val_df.group)&set(test_df.group))
print('Train:',len(train_df),'Validation:',len(val_df),'Held-out test:',len(test_df))
print('Validation class counts:')
print(val_df.groupby('label').size())

## Step 4 — Train a small M0 feasibility pilot
We freeze the pretrained feature extractor and train only the five-class classification head for **one epoch**. This is a *quick hardware/data sanity check*, not the final M0 experiment. The final baseline should be trained/tuned under a fair protocol shared with M1 and M2.

In [ ]:
class CoffeeDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame=frame.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.frame)
    def __getitem__(self,idx):
        r=self.frame.iloc[idx]
        with Image.open(r.path) as im:
            img=ImageOps.exif_transpose(im).convert('RGB')
            img=self.transform(img)
        return img, CLASSES.index(r.label)

norm=transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
train_tf=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),transforms.ToTensor(),norm])
val_tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),norm])
train_loader=DataLoader(CoffeeDataset(train_df,train_tf),batch_size=32,shuffle=True,num_workers=2)
val_loader=DataLoader(CoffeeDataset(val_df,val_tf),batch_size=32,shuffle=False,num_workers=2)

weights=MobileNet_V3_Small_Weights.DEFAULT  # downloads official pretrained weights if not cached
model=mobilenet_v3_small(weights=weights)
for p in model.parameters(): p.requires_grad=False
model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,len(CLASSES))
model=model.to(DEVICE)
optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=1e-3)
loss_fn=nn.CrossEntropyLoss()
start=time.perf_counter()
model.train()
for xb,yb in tqdm(train_loader,desc='Pilot training (1 epoch)'):
    xb,yb=xb.to(DEVICE),yb.to(DEVICE)
    optimizer.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); optimizer.step()
train_seconds=time.perf_counter()-start
print('Training time (s):',round(train_seconds,2))

In [ ]:
model.eval(); preds=[]; targets=[]
with torch.inference_mode():
    for xb,yb in val_loader:
        logits=model(xb.to(DEVICE))
        preds.extend(logits.argmax(dim=1).cpu().tolist())
        targets.extend(yb.tolist())
macro_f1=f1_score(targets,preds,average='macro',labels=list(range(5)),zero_division=0)
acc=accuracy_score(targets,preds)
print('PILOT validation macro-F1:',round(macro_f1,4))
print('PILOT validation accuracy:',round(acc,4))
print(classification_report(targets,preds,labels=list(range(5)),target_names=CLASSES,zero_division=0))

## Step 5 — Measure approximate size and CPU latency
This is **not** a final benchmark. In the final comparison, use the same CPU, thread count, image preprocessing, batch size, warm-up, and runtime for M0/M1/M2. INT8 can be smaller without necessarily being faster.

In [ ]:
import tempfile
cpu_model=model.to('cpu').eval()
with tempfile.TemporaryDirectory() as tmp:
    path=Path(tmp)/'m0_pilot.pth'
    torch.save(cpu_model.state_dict(),path)
    size_mb=path.stat().st_size/1_000_000
n_params=sum(p.numel() for p in cpu_model.parameters())
torch.set_num_threads(min(2,os.cpu_count() or 1))
x=torch.randn(1,3,224,224)
with torch.inference_mode():
    for _ in range(10): cpu_model(x)
    timings=[]
    for _ in range(30):
        t0=time.perf_counter(); cpu_model(x); timings.append((time.perf_counter()-t0)*1000)
report={
    'stage':'M0 pilot only; not final evaluation',
    'sample_images':int(len(audit)),
    'train_images':int(len(train_df)),
    'validation_images':int(len(val_df)),
    'test_images_reserved':int(len(test_df)),
    'train_seconds':round(train_seconds,3),
    'validation_macro_f1':round(float(macro_f1),4),
    'validation_accuracy':round(float(acc),4),
    'parameters':int(n_params),
    'state_dict_size_mb':round(size_mb,3),
    'cpu_latency_median_ms':round(float(np.median(timings)),3),
    'cpu_latency_p95_ms':round(float(np.percentile(timings,95)),3),
    'torch_version':torch.__version__,
    'cuda_available':torch.cuda.is_available(),
    'cpu_threads':torch.get_num_threads(),
    'leakage_warning':'Perceptual hashes do not guarantee independent originals; results are exploratory.'
}
print(json.dumps(report,indent=2))
out=Path('/content/drive/MyDrive/CoffeeDoctor/m0_pilot_results.json')
out.parent.mkdir(parents=True,exist_ok=True)
out.write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Saved pilot measurements to:',out)

## Next steps after this pilot
1. Share the printed class counts and `m0_pilot_results.json` with your team.
2. Audit source-original IDs and augmentation relationships more rigorously before any **final** test score.
3. Confirm the topic with the instructor and co-write Item 1 (Week 5) using your own research interpretation.
4. Next experiment: train the **full M0** fairly, then compare **M1 = MobileNetV3-Small + CBAM**, and **M2 = M1 with INT8 quantization**.
5. Keep RoCoLe severity, coffee/non-coffee verification and the upload demo as separate follow-on modules.

**Course integrity:** The provided draft is AI-assisted planning material; the final submitted writing must comply with the instructor's 25% AI-generated-content limit and citation requirement.